In [1]:
# Run this cell
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
# Here, we load the fuel dataset, and drop any rows that have missing data
vehicle_data = sns.load_dataset('mpg').dropna()
vehicle_data = vehicle_data.sort_values('horsepower', ascending=True)
vehicle_data.head(5)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
19,26.0,4,97.0,46.0,1835,20.5,70,europe,volkswagen 1131 deluxe sedan
102,26.0,4,97.0,46.0,1950,21.0,73,europe,volkswagen super beetle
326,43.4,4,90.0,48.0,2335,23.7,80,europe,vw dasher (diesel)
325,44.3,4,90.0,48.0,2085,21.7,80,europe,vw rabbit c (diesel)
244,43.1,4,90.0,48.0,1985,21.5,78,europe,volkswagen rabbit custom diesel


### For of a simple linear regression, horsepower model

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
model = LinearRegression()

Split out the data into training sets and holdout sets

In [4]:
len(vehicle_data)

392

In [5]:
x_vars = ["horsepower"]

X_train, X_holdout, Y_train, Y_holdout = train_test_split(vehicle_data[x_vars], vehicle_data['mpg'], test_size = 0.25)

In [6]:
len(X_train)

294

In [7]:
from sklearn.model_selection import KFold

Now split data once for each fold, fit a model and calculate the loss for the model's predictions.

In [8]:
kf = KFold(n_splits = 5)

for train_idx, valid_idx in kf.split(X_train):
    
    split_X_valid = X_train.iloc[valid_idx,:]

    print(split_X_valid.head())


     horsepower
144        52.0
334       100.0
236        89.0
90        198.0
79         69.0
     horsepower
291       125.0
176        90.0
205        75.0
311        70.0
350        63.0
     horsepower
15         95.0
340        92.0
58         80.0
59         54.0
250       140.0
     horsepower
160       110.0
31         95.0
132        75.0
345        60.0
246        52.0
     horsepower
2         150.0
185        79.0
186        83.0
396        79.0
339        84.0


kf_split returns two sets of indices, the index for training elements and the index for validation elements. We fit the model on the training split of the data for each fold, predict values on the validation split of the data and calculate the error. 

In [9]:
kf = KFold(n_splits = 5)
validation_errors = np.empty(0)

for train_idx, valid_idx in kf.split(X_train):
    
    split_X_train, split_X_valid = X_train.iloc[train_idx,:], X_train.iloc[valid_idx,:]
    split_Y_train, split_Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx] 
    
    model.fit(split_X_train, split_Y_train)
    
    Y_pred = model.predict(split_X_valid)
    
    validation_errors = np.append(validation_errors, mean_squared_error(split_Y_valid, Y_pred))

In [10]:
validation_errors

array([22.78675197, 28.9336102 , 26.18236437, 22.75830357, 24.10854656])

The average of the validation errors is what we're after

In [11]:
np.mean(validation_errors)

24.953915332331455

### Cross-validation for feature selection

Now select multiple features that we might like to assess

In [12]:
features = vehicle_data[["horsepower", "weight", "acceleration", "model_year","cylinders","displacement","model_year"]]
feature_errors = pd.DataFrame({"feature_name": features.columns})
feature_errors

,feature_name
0,horsepower
1,weight
2,acceleration
3,model_year
4,cylinders
5,displacement
6,model_year


In [13]:
features.head()

,horsepower,weight,acceleration,model_year,cylinders,displacement,model_year
19,46.0,1835,20.5,70,4,97.0,70
102,46.0,1950,21.0,73,4,97.0,73
326,48.0,2335,23.7,80,4,90.0,80
325,48.0,2085,21.7,80,4,90.0,80
244,48.0,1985,21.5,78,4,90.0,78


In [14]:
feature_error = np.empty(0)

for x_var in features.columns:
    X_train, X_holdout, Y_train, Y_holdout = train_test_split(vehicle_data[[x_var]], vehicle_data['mpg'], test_size = 0.25)
    
    kf = KFold(n_splits = 5)
    validation_errors = np.empty(0)

    for train_idx, valid_idx in kf.split(X_train):
    
        split_X_train, split_X_valid = X_train.iloc[train_idx,:], X_train.iloc[valid_idx,:]
        split_Y_train, split_Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx] 
    
        model.fit(split_X_train, split_Y_train)
    
        Y_pred = model.predict(split_X_valid)
    
        validation_errors = np.append(validation_errors, mean_squared_error(split_Y_valid, Y_pred))
        
    feature_error = np.append(feature_error, np.mean(validation_errors))
    

In [15]:
feature_error

array([25.75394873, 19.32390811, 49.56157903, 40.66051803, 22.43790347,
       21.36685301, 41.43937958])

In [16]:
feature_errors["error"] = feature_error
feature_errors

,feature_name,error
0,horsepower,25.753949
1,weight,19.323908
2,acceleration,49.561579
3,model_year,40.660518
4,cylinders,22.437903
5,displacement,21.366853
6,model_year,41.439380


**What features could we build our model from?**

In [17]:
x_vars = ["weight", "displacement","cylinders"]

X_train, X_holdout, Y_train, Y_holdout = train_test_split(vehicle_data[x_vars], vehicle_data['mpg'], test_size = 0.25)

In [18]:
model = LinearRegression()
model.fit(X_train, Y_train)

train_preds = model.predict(X_train)
test_preds = model.predict(X_holdout)

In [19]:
mean_squared_error(Y_train, train_preds)

18.299967880733448

In [20]:
mean_squared_error(Y_holdout, test_preds)

18.400508768952662

### Forward Stepwise Selection

Now that we have the relative quality of each feature, we could do stepwise selection to build an increasingly complex model. If we do this beginning with the best predictor and gradually adding the next best and so on, we are doing **forward** stepwise selection.

In [21]:
feature_importance = feature_errors.sort_values("error", ascending = True)["feature_name"]
feature_importance

1          weight
5    displacement
4       cylinders
0      horsepower
3      model_year
6      model_year
2    acceleration
Name: feature_name, dtype: object

In [22]:
feature_error = np.empty(0)
predictors = []

for i in np.arange(0, len(feature_importance)):
    X_train, X_holdout, Y_train, Y_holdout = train_test_split(vehicle_data[np.array(feature_importance[0:i+1])], vehicle_data['mpg'], test_size = 0.25)

    predictors.append(feature_importance[0:i+1].values)
    
    kf = KFold(n_splits = 5)
    validation_errors = np.empty(0)

    for train_idx, valid_idx in kf.split(X_train):
    
        split_X_train, split_X_valid = X_train.iloc[train_idx,:], X_train.iloc[valid_idx,:]
        split_Y_train, split_Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx] 
    
        model.fit(split_X_train, split_Y_train)
    
        Y_pred = model.predict(split_X_valid)
    
        validation_errors = np.append(validation_errors, mean_squared_error(split_Y_valid, Y_pred))
        
    feature_error = np.append(feature_error, np.mean(validation_errors))
    

In [ ]:
pd.DataFrame({"subset":predictors, "validation error": feature_error})